In [1]:
import struct
import os

ROM_DIR = "/home/jeff/bastel/roms/hohner/ms4"

# Load XE9L firmware
xe9l = open(os.path.join(ROM_DIR, "xe9l_v141.bin"), "rb").read()
xe9r = open(os.path.join(ROM_DIR, "xe9r_v141.bin"), "rb").read()

print(f"XE9L: {len(xe9l)} bytes")
print(f"XE9R: {len(xe9r)} bytes")

# Known program string addresses from Ghidra
known_prog_addrs = [
    0x16A8, 0x1DD8, 0x1EAD, 0x2552, 0x294E, 0x29AA, 0x2AA9, 0x2B05,
    0x2B78, 0x2CBE, 0x2D05, 0x2DDA, 0x2E68, 0x2EAF, 0x3175, 0x356A,
    0x35CD, 0x8600, 0x875E, 0x878E, 0x881B, 0x89A0, 0x89FC, 0x8A2A,
    0x8A5B, 0x8A8C, 0x8AEE, 0x8B1F, 0x8CD8,
]

# Search for a pointer table: look for two consecutive known addresses as BE 16-bit
# E.g. 16 A8 followed by another known address somewhere within ~200 bytes
for addr in known_prog_addrs[:5]:
    hi = (addr >> 8) & 0xFF
    lo = addr & 0xFF
    # Search for this 2-byte sequence in the ROM
    needle = bytes([hi, lo])
    pos = 0
    occurrences = []
    while True:
        pos = xe9l.find(needle, pos)
        if pos == -1:
            break
        occurrences.append(pos)
        pos += 1
    print(f"\n0x{addr:04X} ({xe9l[addr:addr+8].decode('ascii', errors='replace')}): "
          f"found BE ptr at offsets: {[f'0x{o:04X}' for o in occurrences]}")

XE9L: 65536 bytes
XE9R: 65536 bytes

0x16A8 (CONTRA  ): found BE ptr at offsets: ['0x33FB']

0x1DD8 (MUSETC3 ): found BE ptr at offsets: ['0x33DB']

0x1EAD (MUSETA4 ): found BE ptr at offsets: ['0x34E9']

0x2552 (MARIMB  ): found BE ptr at offsets: ['0x33C5']

0x294E (SPACE   ): found BE ptr at offsets: []


In [2]:
# Look at the area around where pointers were found
# Read 0x3380 - 0x3520 to see the table structure
start = 0x3380
length = 0x200
data = xe9l[start:start+length]

print(f"Memory at 0x{start:04X} - 0x{start+length:04X}:")
for i in range(0, len(data), 16):
    addr = start + i
    hex_bytes = ' '.join(f'{b:02X}' for b in data[i:i+16])
    ascii_str = ''.join(chr(b) if 32 <= b < 127 else '.' for b in data[i:i+16])
    print(f"  {addr:04X}: {hex_bytes}  {ascii_str}")


Memory at 0x3380 - 0x3580:
  3380: 00 00 00 00 00 00 84 18 44 02 04 18 FF FF 07 18  ........D.......
  3390: 00 00 00 38 16 00 7F 01 08 00 00 40 00 38 38 38  ...8.......@.888
  33A0: 18 00 04 00 19 F7 FF 03 20 80 C0 03 FF 17 60 1A  ........ .....`.
  33B0: 9C 00 1E 2B 78 22 B1 21 0F 26 AD 2B 05 00 1E 00  ...+x".!.&.+....
  33C0: 1E 00 1E 23 0D 25 52 23 B0 00 1E 00 1E 00 1E 00  ...#.%R#........
  33D0: 1E 00 1E 00 1E 00 1E 1E F4 00 1E 1D D8 27 C1 00  .............'..
  33E0: 1E 27 65 24 0C 2A A9 29 F1 00 1E 00 1E 14 24 13  .'e$.*.)......$.
  33F0: 12 13 C8 12 4C 16 4C 12 B7 17 04 16 A8 00 1E 00  ....L.L.........
  3400: 1E 00 1E 00 1E 00 1E 00 1E 26 51 00 1E 00 1E 00  .........&Q.....
  3410: 1E 00 1E 2A 4D 00 1E 00 1E 00 1E 00 1E 00 1E 00  ...*M...........
  3420: 1E 14 DC 00 1E 00 1E 00 1E 00 1E 00 1E 00 1E 00  ................
  3430: 1E 00 1E 00 1E 00 1E 00 1E 00 1E 00 1E 00 1E 00  ................
  3440: 1E 00 1E 00 1E 00 1E 00 1E 00 1E 00 1E 00 1E 00  ................
  3450: 1E 

In [3]:
# The table appears to be near 0x33AD. Let me verify by reading program names
# at the addresses stored in the table.

def read_be16(data, offset):
    return (data[offset] << 8) | data[offset + 1]

def read_le16(data, offset):
    return data[offset] | (data[offset + 1] << 8)

def get_program_name(rom, addr):
    """Read 8-byte program name at addr"""
    if addr + 9 > len(rom) or addr < 0:
        return None
    name = rom[addr:addr+8]
    try:
        return name.decode('ascii').rstrip('\x00 ')
    except:
        return None

# Try table starting at 0x33AD, 128 entries of 2-byte BE
table_start = 0x33AD
n_entries = 128

print(f"Program pointer table at 0x{table_start:04X} ({n_entries} entries):\n")
print(f"{'Idx':>4} {'TblAddr':>8} {'ProgAddr':>9} {'Name':<12} {'Flags':>6} {'ARAM ptr':>9}")
print("-" * 60)

valid_count = 0
for i in range(n_entries):
    tbl_addr = table_start + i * 2
    prog_addr = read_be16(xe9l, tbl_addr)
    name = get_program_name(xe9l, prog_addr)
    
    if name and len(name) > 0 and name != "SILENCE" and all(c.isalnum() or c in ' _-' for c in name):
        flags = xe9l[prog_addr + 9]
        aram_ptr = read_le16(xe9l, prog_addr + 10)
        print(f"{i:4d} 0x{tbl_addr:04X}  0x{prog_addr:04X}  {name:<12} 0x{flags:02X}   0x{aram_ptr:04X}")
        valid_count += 1

print(f"\n{valid_count} non-SILENCE programs found")


Program pointer table at 0x33AD (128 entries):

 Idx  TblAddr  ProgAddr Name          Flags  ARAM ptr
------------------------------------------------------------
   0 0x33AD  0x1760  BDCLA0       0x11   0x006A
   1 0x33AF  0x1A9C  SBDCLA0      0x11   0x006A
   3 0x33B3  0x2B78  weste3       0x11   0x006A
   4 0x33B5  0x22B1  DXPIA1       0x11   0x006A
   5 0x33B7  0x210F  EPIANO2      0x11   0x002A
   6 0x33B9  0x26AD  HARPSI1      0x11   0x006A
   7 0x33BB  0x2B05  CLAVINET     0x11   0x00EA
  11 0x33C3  0x230D  JVIBES       0x11   0x006A
  12 0x33C5  0x2552  MARIMB       0x11   0x006A
  13 0x33C7  0x23B0  XYLO         0x11   0x006A
  21 0x33D7  0x1EF4  ACCA2        0x11   0x00AA
  23 0x33DB  0x1DD8  MUSETC3      0x11   0x00AA
  24 0x33DD  0x27C1  NYGTA2       0x11   0x006A
  26 0x33E1  0x2765  JZGTF2       0x11   0x006A
  27 0x33E3  0x240C  ELGTE2       0x11   0x006A
  28 0x33E5  0x2AA9  MUGUIT       0x11   0x006A
  29 0x33E7  0x29F1  ACGUIT       0x11   0x006A
  32 0x33ED  0x1424  

In [4]:
import sys
sys.path.insert(0, '/home/jeff/bastel/mame/mame/sam8905')
from sam8905_aram_decoder import decode_instruction, format_instruction, decode_algorithm, analyze_dram_usage

# Extract all 4 algorithms from XE9L
alg_addrs = [0x002A, 0x006A, 0x00AA, 0x00EA]

algorithms = {}
for addr in alg_addrs:
    words = []
    for i in range(32):
        lo = xe9l[addr + i*2]
        hi = xe9l[addr + i*2 + 1]
        words.append((hi << 8) | lo)
    algorithms[addr] = words

# Check each algorithm for WWE instructions
# WWE fires when: RSP (emitter_sel==3) AND clearB (bit2=0) AND WSP (bit8=1)
# Test: (inst & 0x704) == 0x700

print("=" * 80)
print("XE9L A-RAM ALGORITHMS - WWE (Write Waveform External) Analysis")
print("=" * 80)

for addr, words in algorithms.items():
    # Count programs using this algorithm
    users = []
    for i in range(128):
        prog_addr = read_be16(xe9l, 0x33AD + i*2)
        name = get_program_name(xe9l, prog_addr)
        if name and name != "SILENCE":
            aram_ptr = read_le16(xe9l, prog_addr + 10)
            if aram_ptr == addr:
                users.append(name)

    print(f"\n{'─' * 70}")
    print(f"Algorithm at 0x{addr:04X} ({len(users)} programs: {', '.join(users[:6])}"
          f"{'...' if len(users) > 6 else ''})")
    print(f"{'─' * 70}")
    
    wwe_found = False
    for pc, raw in enumerate(words):
        inst = decode_instruction(raw)
        formatted = format_instruction(inst)
        
        # Check for WWE: RSP + clearB + WSP
        is_wwe = (raw & 0x704) == 0x700
        marker = ""
        if inst.wsp:
            marker = " ***"
        if is_wwe:
            marker = " <<< WWE"
            wwe_found = True
        
        print(f"  PC{pc:02d}: {formatted}{marker}")
    
    if not wwe_found:
        print(f"\n  >> No WWE instructions found")
    else:
        print(f"\n  >> WWE instructions present - uses external memory write!")


XE9L A-RAM ALGORITHMS - WWE (Write Waveform External) Analysis

──────────────────────────────────────────────────────────────────────
Algorithm at 0x002A (3 programs: EPIANO2, B16_8, B8)
──────────────────────────────────────────────────────────────────────
  PC00: 79F7  RM 15, <WXY, WSP> ***
  PC01: 396F  RM 7, <WA, WPHI, WSP> ***
  PC02: 30BF  RM 6, <WB>
  PC03: 3ADF  RADD 7, <WM>
  PC04: 7077  RM 14, <WA, WXY>
  PC05: 296F  RM 5, <WA, WPHI, WSP> ***
  PC06: 7CBF  RP, <WB>
  PC07: 7ABF  RADD, <WB>
  PC08: 207F  RM 4, <WA>
  PC09: 2ADF  RADD 5, <WM>
  PC10: 60B7  RM 12, <WB, WXY>
  PC11: 687F  RM 13, <WA>
  PC12: 639F  RADD 12, <WB, WM, WSP> ***
  PC13: 196E  RM 3, <WA, WPHI, WACC, WSP> ***
  PC14: 10BF  RM 2, <WB>
  PC15: 1ADF  RADD 3, <WM>
  PC16: 50B7  RM 10, <WB, WXY>
  PC17: 587F  RM 11, <WA>
  PC18: 539F  RADD 10, <WB, WM, WSP> ***
  PC19: 096F  RM 1, <WA, WPHI, WSP> ***
  PC20: 7CBF  RP, <WB>
  PC21: 7ABF  RADD, <WB>
  PC22: 007F  RM 0, <WA>
  PC23: 0ADF  RADD 1, <WM>
  PC24: 

In [6]:
# Re-examine the WWE trigger mechanism:
# Per prog guide: /WWE goes low during a 3-instruction clearB sequence:
#   <clearB, WSP>, <clearB>, <clearB>
# /WWE is LOW on the 3rd clearB instruction
# 
# But we want /WWE low while WD bus is readable (high-Z from SAM side)
# The 74HC244 buffer puts PHI[0:2] onto WD[0:2] when /WWE is low AND WA16=0
# For reading this back INTO the SAM, we'd need WXY or similar receiver active
# during the /WWE-low cycle
#
# Let me search for clearB sequences (not just isolated clearB instructions)
# clearB active = bit 2 is 0

def find_clearb_sequences(words, min_length=2):
    """Find runs of consecutive instructions with clearB active"""
    sequences = []
    run_start = None
    run_length = 0
    
    for pc, w in enumerate(words):
        clearb = not (w & 0x04)  # bit 2 = 0 means clearB active
        if clearb:
            if run_start is None:
                run_start = pc
                run_length = 1
            else:
                run_length += 1
        else:
            if run_start is not None and run_length >= min_length:
                sequences.append((run_start, run_length, words[run_start:run_start+run_length]))
            run_start = None
            run_length = 0
    
    if run_start is not None and run_length >= min_length:
        sequences.append((run_start, run_length, words[run_start:run_start+run_length]))
    
    return sequences

print("=" * 70)
print("XE9L: Searching for clearB sequences (min 2 consecutive)")
print("=" * 70)

for addr, words in algorithms.items():
    seqs = find_clearb_sequences(words, min_length=2)
    # Also count single clearB instructions
    single_clearb = [(pc, w) for pc, w in enumerate(words) if not (w & 0x04)]
    
    print(f"\nAlgorithm 0x{addr:04X}:")
    print(f"  Single clearB instructions: {len(single_clearb)}")
    for pc, w in single_clearb:
        inst = decode_instruction(w)
        print(f"    PC{pc:02d}: {format_instruction(inst)}")
    
    if seqs:
        for start, length, seq_words in seqs:
            print(f"  Sequence at PC{start:02d}-PC{start+length-1:02d} ({length} instructions):")
            for i, w in enumerate(seq_words):
                inst = decode_instruction(w)
                wsp = "WSP" if inst.wsp else ""
                print(f"    PC{start+i:02d}: {format_instruction(inst)}")
    else:
        print(f"  No clearB sequences found")

# Now check XE9R
print("\n" + "=" * 70)
print("XE9R: Extracting algorithms and searching for clearB sequences")
print("=" * 70)

# Check if XE9R has same algorithm layout
# First verify algorithms at same addresses
xe9r_algs = {}
for addr in alg_addrs:
    words = []
    for i in range(32):
        lo = xe9r[addr + i*2]
        hi = xe9r[addr + i*2 + 1]
        words.append((hi << 8) | lo)
    xe9r_algs[addr] = words
    
    # Compare with XE9L
    same = (words == algorithms[addr])
    single_clearb = [(pc, w) for pc, w in enumerate(words) if not (w & 0x04)]
    seqs = find_clearb_sequences(words, min_length=2)
    
    print(f"\nAlgorithm 0x{addr:04X}: {'SAME as XE9L' if same else 'DIFFERENT from XE9L'}")
    print(f"  Single clearB instructions: {len(single_clearb)}")
    for pc, w in single_clearb:
        inst = decode_instruction(w)
        print(f"    PC{pc:02d}: {format_instruction(inst)}")
    if seqs:
        for start, length, seq_words in seqs:
            print(f"  >>> Sequence at PC{start:02d}-PC{start+length-1:02d} ({length} instructions):")
            for i, w in enumerate(seq_words):
                inst = decode_instruction(w)
                print(f"    PC{start+i:02d}: {format_instruction(inst)}")


XE9L: Searching for clearB sequences (min 2 consecutive)

Algorithm 0x002A:
  Single clearB instructions: 0
  No clearB sequences found

Algorithm 0x006A:
  Single clearB instructions: 0
  No clearB sequences found

Algorithm 0x00AA:
  Single clearB instructions: 1
    PC01: 7EFB  RSP, <clearB>
  No clearB sequences found

Algorithm 0x00EA:
  Single clearB instructions: 0
  No clearB sequences found

XE9R: Extracting algorithms and searching for clearB sequences

Algorithm 0x002A: DIFFERENT from XE9L
  Single clearB instructions: 0

Algorithm 0x006A: DIFFERENT from XE9L
  Single clearB instructions: 1
    PC01: 7EFB  RSP, <clearB>

Algorithm 0x00AA: DIFFERENT from XE9L
  Single clearB instructions: 0

Algorithm 0x00EA: DIFFERENT from XE9L
  Single clearB instructions: 22
    PC02: 0182  RM 0, <WB, WM, WPHI, WXY, clearB, WACC, WSP>
    PC03: 7300  RADD 14, <WA, WB, WM, WPHI, WXY, clearB, WWF, WACC, WSP>
    PC05: FFF8  RSP, <clearB, WWF, WACC, WSP>
    PC07: 0002  RM 0, <WA, WB, WM, WPH

In [7]:
# Full decode of XE9R algorithm 0x00EA (the one with WWE)
print("=" * 70)
print("XE9R Algorithm 0x00EA - Full Decode (FX/Reverb with WWE)")
print("=" * 70)

words = xe9r_algs[0x00EA]
for pc, raw in enumerate(words):
    inst = decode_instruction(raw)
    formatted = format_instruction(inst)
    
    # Check clearB
    clearb = not (raw & 0x04)
    wsp = bool(raw & 0x100)
    
    notes = []
    if clearb and wsp:
        notes.append("clearB+WSP")
    elif clearb:
        notes.append("clearB")
    
    # Check for WWE trigger: 3rd+ consecutive clearB 
    note_str = f"  [{', '.join(notes)}]" if notes else ""
    print(f"PC{pc:02d}: {formatted}{note_str}")

# Also show the raw hex
print(f"\nRaw hex (LE words):")
for i in range(0, 32, 8):
    chunk = words[i:i+8]
    print(f"  " + ', '.join(f'0x{w:04X}' for w in chunk))

# Compare all 4 algorithms between XE9L and XE9R
print(f"\n{'=' * 70}")
print("Algorithm comparison XE9L vs XE9R")
print(f"{'=' * 70}")
for addr in alg_addrs:
    same = (algorithms[addr] == xe9r_algs[addr])
    print(f"  0x{addr:04X}: {'IDENTICAL' if same else 'DIFFERENT'}")


XE9R Algorithm 0x00EA - Full Decode (FX/Reverb with WWE)
PC00: F87F  RM 15, <WA>
PC01: 4BFF  RADD, <WSP>
PC02: 0182  RM 0, <WB, WM, WPHI, WXY, clearB, WACC, WSP>  [clearB+WSP]
PC03: 7300  RADD 14, <WA, WB, WM, WPHI, WXY, clearB, WWF, WACC, WSP>  [clearB+WSP]
PC04: 7F1D  RSP, <WA, WB, WM, WWF, WSP>
PC05: FFF8  RSP, <clearB, WWF, WACC, WSP>  [clearB+WSP]
PC06: D236  RADD, <WA, WB, WXY, WACC>
PC07: 0002  RM 0, <WA, WB, WM, WPHI, WXY, clearB, WACC>  [clearB]
PC08: 2693  RSP, <WB, WM, WXY, clearB>  [clearB]
PC09: 507F  RM 10, <WA>
PC10: 5D4F  RP 11, <WA, WM, WPHI, WSP>
PC11: 0182  RM 0, <WB, WM, WPHI, WXY, clearB, WACC, WSP>  [clearB+WSP]
PC12: FB00  RADD 15, <WA, WB, WM, WPHI, WXY, clearB, WWF, WACC, WSP>  [clearB+WSP]
PC13: 7F09  RSP, <WA, WB, WM, WPHI, clearB, WWF, WSP>  [clearB+WSP]
PC14: 3C88  RP 7, <WB, WM, WPHI, clearB, WWF, WACC>  [clearB]
PC15: 825B  RADD 0, <WA, WM, clearB>  [clearB]
PC16: 0001  RM 0, <WA, WB, WM, WPHI, WXY, clearB, WWF>  [clearB]
PC17: 09FB  RM 1, <clearB, WSP>  

In [8]:
# Check XE9R firmware structure from the start
print("XE9R firmware start (0x0000-0x0050):")
for i in range(0, 0x50, 16):
    hex_bytes = ' '.join(f'{b:02X}' for b in xe9r[i:i+16])
    ascii_str = ''.join(chr(b) if 32 <= b < 127 else '.' for b in xe9r[i:i+16])
    print(f"  {i:04X}: {hex_bytes}  {ascii_str}")

# Check if SILENCE string is at same position
silence_pos = xe9r.find(b'SILENCE ')
print(f"\n'SILENCE ' string at: 0x{silence_pos:04X}" if silence_pos >= 0 else "\n'SILENCE ' not found")

# Check if any of the first programs from XE9L string list exist in XE9R
for name in [b'CONTRA', b'MARIMB', b'EPIANO2', b'BDCLA0', b'STA DRUM']:
    pos = xe9r.find(name)
    if pos >= 0:
        print(f"'{name.decode()}' found at 0x{pos:04X}")
    else:
        print(f"'{name.decode()}' NOT found in XE9R")


XE9R firmware start (0x0000-0x0050):
  0000: 02 95 3E C2 AF C2 AB C2 A8 02 8D 92 25 E0 50 02  ..>.........%.P.
  0010: 05 83 25 82 50 02 05 83 F5 82 22 02 8D 76 53 49  ..%.P....."..vSI
  0020: 4C 45 4E 43 45 20 00 00 00 00 F5 79 EF 30 F7 60  LENCE .....y.0.`
  0030: 7F 58 BF 7C CF 5A F7 38 EF 11 FD 20 6E 08 BF 68  .X.|.Z.8... n..h
  0040: DF 4A 3F 11 DF 52 BF 18 DF 53 F7 28 EF 51 FD 20  .J?..R...S.(.Q. 

'SILENCE ' string at: 0x001E
'CONTRA' NOT found in XE9R
'MARIMB' NOT found in XE9R
'EPIANO2' NOT found in XE9R
'BDCLA0' NOT found in XE9R
'STA DRUM' NOT found in XE9R


In [9]:
# Find algorithm boundaries in XE9R by looking for the start (after SILENCE)
# and tracking 32-word blocks ending with 0x7FFF NOPs

def find_algorithms(rom, start_addr=0x002A, max_search=0x400):
    """Find 32-word algorithm blocks starting from start_addr"""
    algs = []
    addr = start_addr
    
    while addr + 64 <= len(rom) and addr < start_addr + max_search:
        # Read 32 LE words
        words = []
        for i in range(32):
            lo = rom[addr + i*2]
            hi = rom[addr + i*2 + 1]
            words.append((hi << 8) | lo)
        
        # Check if this looks like an algorithm (ends with 0x7FFF NOPs 
        # and all values <= 0x7FFF i.e. 15-bit)
        all_valid = all(w <= 0x7FFF for w in words)
        has_nop_tail = words[-1] == 0x7FFF
        
        if all_valid and has_nop_tail:
            # Count active (non-NOP) instructions
            active = sum(1 for w in words if w != 0x7FFF)
            algs.append((addr, words, active))
            addr += 64  # next potential algorithm
        else:
            break  # no more algorithms in sequence
    
    return algs

print("XE9L Algorithms:")
xe9l_algs_found = find_algorithms(xe9l, 0x002A)
for addr, words, active in xe9l_algs_found:
    print(f"  0x{addr:04X}: {active} active instructions, first={words[0]:04X}")

print(f"\nXE9R Algorithms:")
xe9r_algs_found = find_algorithms(xe9r, 0x002A)
for addr, words, active in xe9r_algs_found:
    print(f"  0x{addr:04X}: {active} active instructions, first={words[0]:04X}")

# Now check for clearB sequences in XE9R algorithms at CORRECT addresses
print(f"\n{'=' * 70}")
print("XE9R algorithms - clearB sequence analysis")
print(f"{'=' * 70}")

for addr, words, active in xe9r_algs_found:
    single_clearb = [(pc, w) for pc, w in enumerate(words) if not (w & 0x04)]
    seqs = find_clearb_sequences(words, min_length=2)
    
    print(f"\nAlgorithm at 0x{addr:04X} ({active} active instructions):")
    print(f"  clearB instructions: {len(single_clearb)}")
    if seqs:
        for start, length, seq_words in seqs:
            print(f"  >>> clearB sequence PC{start:02d}-PC{start+length-1:02d} ({length} instr):")
            for i, w in enumerate(seq_words):
                inst = decode_instruction(w)
                wsp = " +WSP" if inst.wsp else ""
                print(f"      PC{start+i:02d}: {format_instruction(inst)}{wsp}")
    else:
        print(f"  No clearB sequences")


XE9L Algorithms:
  0x002A: 28 active instructions, first=79F7
  0x006A: 29 active instructions, first=79F5
  0x00AA: 30 active instructions, first=08EF
  0x00EA: 30 active instructions, first=296F

XE9R Algorithms:
  0x002A: 29 active instructions, first=79F5
  0x006A: 30 active instructions, first=08EF
  0x00AA: 30 active instructions, first=09EF

XE9R algorithms - clearB sequence analysis

Algorithm at 0x002A (29 active instructions):
  clearB instructions: 0
  No clearB sequences

Algorithm at 0x006A (30 active instructions):
  clearB instructions: 1
  No clearB sequences

Algorithm at 0x00AA (30 active instructions):
  clearB instructions: 0
  No clearB sequences


In [10]:
# Find the XE9R program table and check what A-RAM pointers programs use
# First, find XE9R program names
print("XE9R defined strings (searching for 8-char padded names):")
xe9r_progs = []
for addr in range(0x0100, len(xe9r) - 12):
    # Check for 8-byte ASCII name + null terminator pattern
    name_bytes = xe9r[addr:addr+8]
    null = xe9r[addr+8]
    if null != 0:
        continue
    # Check if all name bytes are printable ASCII
    if all(32 <= b < 127 for b in name_bytes):
        name = name_bytes.decode('ascii').rstrip()
        if len(name) >= 2 and name[0].isalpha():
            # Check flags byte looks reasonable
            flags = xe9r[addr+9]
            slot_count = flags & 0x0F
            if slot_count > 0 and slot_count <= 8:
                aram_ptr = read_le16(xe9r, addr + 10)
                if aram_ptr < 0x4000:  # reasonable ROM address
                    xe9r_progs.append((addr, name, flags, aram_ptr))

print(f"Found {len(xe9r_progs)} potential programs")
print(f"\n{'Addr':>7} {'Name':<12} {'Flags':>6} {'ARAM':>7}")
print("-" * 40)

# Collect unique A-RAM pointers
aram_ptrs = set()
for addr, name, flags, aram_ptr in xe9r_progs[:50]:
    aram_ptrs.add(aram_ptr)
    print(f"0x{addr:04X}  {name:<12} 0x{flags:02X}   0x{aram_ptr:04X}")

print(f"\nUnique A-RAM pointers: {sorted(f'0x{p:04X}' for p in aram_ptrs)}")


XE9R defined strings (searching for 8-char padded names):
Found 111 potential programs

   Addr Name          Flags    ARAM
----------------------------------------
0x0BF1  STRGA2       0x11   0x002A
0x0C4B  STRGE3       0x11   0x002A
0x0CA5  STRGE4       0x11   0x006A
0x0CEC  STRGE5       0x11   0x006A
0x0D33  STRG2A2      0x01   0x002A
0x0D8D  STRG2E3      0x01   0x002A
0x0DE7  STRG2E4      0x01   0x002A
0x0E41  STRG2E5      0x01   0x006A
0x0E88  SYNTSTR      0x11   0x002A
0x0EE2  SYNTSTRH     0x11   0x006A
0x0F29  ORCHA2       0x11   0x002A
0x0F85  ORCHE3       0x11   0x002A
0x0FE1  ORCHE4       0x11   0x002A
0x103D  ORCHE5       0x11   0x006A
0x1084  PIZZF2       0x11   0x002A
0x10E0  PIZZF3       0x11   0x002A
0x113C  PIZZF4       0x11   0x002A
0x1198  MIXB2        0x11   0x002A
0x11F2  MIXF3        0x11   0x002A
0x124C  MIXB3H       0x11   0x006A
0x1293  MANB2        0x11   0x002A
0x12ED  MANF3        0x11   0x002A
0x1347  MANB3H       0x11   0x006A
0x138E  DOUA2        0x11   0x

In [11]:
# Check ALL unique A-RAM pointers across all 111 XE9R programs
all_aram_ptrs = set()
for addr, name, flags, aram_ptr in xe9r_progs:
    all_aram_ptrs.add(aram_ptr)

print(f"All unique A-RAM pointers in XE9R (across {len(xe9r_progs)} programs):")
for p in sorted(all_aram_ptrs):
    count = sum(1 for _, _, _, ap in xe9r_progs if ap == p)
    print(f"  0x{p:04X}: {count} programs")

# Decode the unused XE9R algorithm 0x00AA 
print(f"\nXE9R Algorithm 0x00AA (not referenced by any program):")
words = xe9r_algs[0x00AA]
for pc, raw in enumerate(words):
    if raw == 0x7FFF and pc > 20:
        print(f"  PC{pc:02d}: 7FFF  NOP")
    else:
        inst = decode_instruction(raw)
        print(f"  PC{pc:02d}: {format_instruction(inst)}")

# Now search the ENTIRE XE9R ROM for 64-byte blocks of valid A-RAM data
# that contain clearB sequences
print(f"\n{'=' * 70}")
print("Scanning entire XE9R ROM for algorithm-like blocks with clearB")
print(f"{'=' * 70}")

found = []
for scan_addr in range(0, len(xe9r) - 64, 2):  # step by 2 for word alignment
    words = []
    valid = True
    for i in range(32):
        lo = xe9r[scan_addr + i*2]
        hi = xe9r[scan_addr + i*2 + 1]
        w = (hi << 8) | lo
        if w > 0x7FFF:
            valid = False
            break
        words.append(w)
    
    if not valid:
        continue
    
    # Must end with at least one 0x7FFF NOP
    if words[-1] != 0x7FFF:
        continue
    
    # Must have at least 10 active (non-NOP) instructions
    active = sum(1 for w in words if w != 0x7FFF)
    if active < 10:
        continue
    
    # Check for clearB sequences (3+ consecutive)
    seqs = find_clearb_sequences(words, min_length=3)
    if seqs:
        found.append((scan_addr, words, active, seqs))

if found:
    for scan_addr, words, active, seqs in found:
        print(f"\n  Block at 0x{scan_addr:04X} ({active} active instructions):")
        for start, length, seq_words in seqs:
            print(f"    clearB sequence PC{start:02d}-PC{start+length-1:02d} ({length} instructions)")
else:
    print("\n  No algorithm-like blocks with 3+ consecutive clearB found in XE9R ROM!")


All unique A-RAM pointers in XE9R (across 111 programs):
  0x002A: 71 programs
  0x0063: 1 programs
  0x006A: 34 programs
  0x00AA: 4 programs
  0x31F7: 1 programs

XE9R Algorithm 0x00AA (not referenced by any program):
  PC00: 09EF  RM 1, <WPHI, WSP>
  PC01: 01F5  RM 0, <WXY, WWF, WSP>
  PC02: 206F  RM 4, <WA, WPHI>
  PC03: 18BF  RM 3, <WB>
  PC04: 22DF  RADD 4, <WM>
  PC05: 093F  RM 1, <WA, WB, WSP>
  PC06: 0ADF  RADD 1, <WM>
  PC07: 10BF  RM 2, <WB>
  PC08: 0BDF  RADD 1, <WM, WSP>
  PC09: 28F7  RM 5, <WXY>
  PC10: 60FD  RM 12, <WWF>
  PC11: 38EF  RM 7, <WPHI>
  PC12: 7C7F  RP, <WA>
  PC13: 40F7  RM 8, <WXY>
  PC14: 48BF  RM 9, <WB>
  PC15: 7A7F  RADD, <WA>
  PC16: 7CBF  RP, <WB>
  PC17: 30EF  RM 6, <WPHI>
  PC18: 78FD  RM 15, <WWF>
  PC19: 7AF7  RADD, <WXY>
  PC20: 407F  RM 8, <WA>
  PC21: 7CBF  RP, <WB>
  PC22: 60FD  RM 12, <WWF>
  PC23: 42D7  RADD 8, <WM, WXY>
  PC24: 487F  RM 9, <WA>
  PC25: 7CBF  RP, <WB>
  PC26: 4ACF  RADD 9, <WM, WPHI>
  PC27: 50B7  RM 10, <WB, WXY>
  PC28: 58

In [12]:
# Check anomalous A-RAM pointers
for addr, name, flags, aram_ptr in xe9r_progs:
    if aram_ptr not in [0x002A, 0x006A, 0x00AA]:
        print(f"Anomalous: {name} at 0x{addr:04X} -> A-RAM ptr 0x{aram_ptr:04X}")
        if aram_ptr + 64 < len(xe9r):
            words = []
            for i in range(32):
                lo = xe9r[aram_ptr + i*2]
                hi = xe9r[aram_ptr + i*2 + 1]
                words.append((hi << 8) | lo)
            print(f"  First 4 words: {', '.join(f'0x{w:04X}' for w in words[:4])}")
            print(f"  All valid 15-bit: {all(w <= 0x7FFF for w in words)}")

# Also do same scan for XE9L
print(f"\n{'=' * 70}")
print("XE9L: Full ROM scan for algorithm blocks with clearB sequences")
print(f"{'=' * 70}")

found_l = []
for scan_addr in range(0, len(xe9l) - 64, 2):
    words = []
    valid = True
    for i in range(32):
        lo = xe9l[scan_addr + i*2]
        hi = xe9l[scan_addr + i*2 + 1]
        w = (hi << 8) | lo
        if w > 0x7FFF:
            valid = False
            break
        words.append(w)
    
    if not valid:
        continue
    if words[-1] != 0x7FFF:
        continue
    active = sum(1 for w in words if w != 0x7FFF)
    if active < 10:
        continue
    
    seqs = find_clearb_sequences(words, min_length=3)
    if seqs:
        found_l.append((scan_addr, words, active, seqs))

if found_l:
    for scan_addr, words, active, seqs in found_l:
        print(f"\n  Block at 0x{scan_addr:04X} ({active} active):")
        for start, length, seq_words in seqs:
            print(f"    clearB seq PC{start}-PC{start+length-1} ({length} instr)")
else:
    print("  No algorithm-like blocks with 3+ consecutive clearB in XE9L ROM either!")

# Also check: are there ANY 3+ consecutive clearB sequences in either ROM,
# even outside of proper algorithm blocks?
print(f"\n{'=' * 70}")
print("Raw clearB byte scan (bit pattern in LE word pairs)")
print(f"{'=' * 70}")
for label, rom in [("XE9L", xe9l), ("XE9R", xe9r)]:
    max_run = 0
    max_run_addr = 0
    run = 0
    run_start = 0
    for addr in range(0, len(rom) - 2, 2):
        w = rom[addr] | (rom[addr+1] << 8)
        if w <= 0x7FFF and not (w & 0x04):
            if run == 0:
                run_start = addr
            run += 1
        else:
            if run > max_run:
                max_run = run
                max_run_addr = run_start
            run = 0
    print(f"  {label}: longest consecutive clearB run = {max_run} words at 0x{max_run_addr:04X}")


Anomalous: rflute at 0x325E -> A-RAM ptr 0x31F7
  First 4 words: 0x79F7, 0x48FD, 0x186F, 0x10BF
  All valid 15-bit: True
Anomalous: RFLUTE at 0x32C1 -> A-RAM ptr 0x0063
  First 4 words: 0xFF54, 0xFF7F, 0xFF7F, 0xEF7F
  All valid 15-bit: False

XE9L: Full ROM scan for algorithm blocks with clearB sequences
  No algorithm-like blocks with 3+ consecutive clearB in XE9L ROM either!

Raw clearB byte scan (bit pattern in LE word pairs)
  XE9L: longest consecutive clearB run = 231 words at 0x7816
  XE9R: longest consecutive clearB run = 152 words at 0x750C


In [5]:
# Check if there's a second program table (drum programs at 0x8600+)
# From Ghidra strings: STA DRUM @ 0x8600, MONDO @ 0x878E, etc.
# Looking at the area around 0x34AF which had pointer-like values

print("Second table area (0x34AF):")
print(f"{'Idx':>4} {'TblAddr':>8} {'ProgAddr':>9} {'Name':<12} {'Flags':>6} {'ARAM ptr':>9}")
print("-" * 60)

table2_start = 0x34AF
valid2 = 0
for i in range(64):  # try 64 entries
    tbl_addr = table2_start + i * 2
    if tbl_addr + 1 >= len(xe9l):
        break
    prog_addr = read_be16(xe9l, tbl_addr)
    name = get_program_name(xe9l, prog_addr)
    
    if name and len(name) > 0 and all(c.isalnum() or c in ' _-.' for c in name):
        flags = xe9l[prog_addr + 9]
        aram_ptr = read_le16(xe9l, prog_addr + 10)
        marker = ""
        if name == "SILENCE":
            marker = " (empty)"
        print(f"{i:4d} 0x{tbl_addr:04X}  0x{prog_addr:04X}  {name:<12} 0x{flags:02X}   0x{aram_ptr:04X}{marker}")
        valid2 += 1
    else:
        break  # likely past end of table

print(f"\n{valid2} entries found in second table")


Second table area (0x34AF):
 Idx  TblAddr  ProgAddr Name          Flags  ARAM ptr
------------------------------------------------------------
   0 0x34AF  0x1874  BDCLD2       0x11   0x006A
   1 0x34B1  0x1818  BDCLA2       0x11   0x006A
   2 0x34B3  0x1988  BDD3         0x11   0x006A
   3 0x34B5  0x18D0  BDA3         0x11   0x006A
   4 0x34B7  0x19E4  BDD4         0x11   0x006A
   5 0x34B9  0x192C  BDA4         0x11   0x006A
   6 0x34BB  0x1A40  BDD5         0x11   0x006A
   7 0x34BD  0x1AF8  SBDCLA1      0x11   0x006A
   8 0x34BF  0x1BB0  SBDCLD2      0x11   0x006A
   9 0x34C1  0x1B54  SBDCLA2      0x11   0x006A
  10 0x34C3  0x1CC4  SBDD3        0x11   0x006A
  11 0x34C5  0x1C0C  SBDA3        0x11   0x006A
  12 0x34C7  0x1D20  SBDD4        0x11   0x006A
  13 0x34C9  0x1C68  SBDA4        0x11   0x006A
  14 0x34CB  0x1D7C  SBDD5        0x11   0x006A
  15 0x34CD  0x2BD4  weste4       0x11   0x006A
  16 0x34CF  0x2010  DXPIA2       0x11   0x006A
  17 0x34D1  0x206C  DXPIA3       0x11   

New Notebook Created by Jupyter MCP Server